## Node and edges: 

#### Node Features (Per-Residue Properties)

Rosetta per-residue energy (decomposed from total energy)  
FoldX per-residue energy contribution  
SASA per residue   
Secondary structure assignment (one-hot encoded: helix, sheet, loop)  
B-factors (from crystallographic data)    
Residue identity (one-hot encoded or using amino acid embedding)  
Per-residue electrostatic potential (calculable from PDB using tools like APBS or PDB2PQR)  
Predicted pKa values (calculable using PROPKA or H++)  
Hydrophobicity scores (based on amino acid scales)  
Conservation scores (if you have evolutionary data)  

#### Edge Features (Inter-Residue Interactions)

Hydrogen bonds (binary or strength-weighted)  
Salt bridges (binary or strength-weighted)  
Hydrophobic contacts (binary or distance-weighted)  
Disulfide bonds (binary)  
Distance between residues (spatial proximity)  
Contact map (binary or distance-weighted)  
Sequence separation (|i-j| for residues i and j)  

#### Additional Physics-Based Features Calculable from PDB

- Per-residue electrostatic potential  
Use PDB2PQR followed by APBS  
DelPhiPKa can also calculate these properties  


- Local dielectric constant  
Approximate using SASA and local environment  
Tools like DelPhi can estimate local dielectric properties  


- Solvent exposure at different ionic strengths  
Use APBS with varying ionic strength parameters  
Calculate accessibility changes at different salt concentrations  


- Predicted pKa values  
PROPKA can predict pKa values directly from PDB    
H++ server also provides pKa predictions  
These tools account for the local environment of titratable groups  



### Graph Structure Recommendations

Node definition: Each amino acid residue as a node  
Edge definition: Multiple edge types based on interaction type (hydrogen bond, salt bridge, etc.)  
Cutoff distance: Consider connections between residues within ~10-12Å  
Multi-scale representation: Consider including both residue-level and secondary structure-level graphs  

### Implementation Steps

Use BioPython or similar libraries to parse PDB files  
Calculate basic structural properties (SASA, secondary structure) using tools like DSSP  
Use Rosetta and FoldX to calculate energy terms  
Use PDB2PQR and APBS for electrostatic calculations  
Use PROPKA for pKa predictions  
Construct your graph with calculated features as node and edge attributes  

This comprehensive set of features will provide your GNN with rich information about the protein's structure and energetics, allowing it to learn the relationship between these properties and optimal stability conditions.


NEW update: Protein params can be get with Graphein , calculation params need other tool (i.e. Pyrosetta)

# Constrains

Use relative coordinates instead of absolute ones: Bond lengths, angles and torsions.   

This allows translation and rotation invariance, energy becomes indipendent of the protein's positi9on and orientation in space --> more robust model.   

Proposed Loss terms:   
- Bond loss (bond stretching): deviation from the ideal bond length 
- Angle loss (angle bending): using the relative position of the atoms involved in the angle. 
- Torsion Loss: takes 4 atoms but using relative coordinates removed a term for the protein position/orientation. 
- Non-bonded interactions: Van der Waals and Lennard Jones potential law; ensure the  minimization of energy and attraction but accounting for sterical hyndrance. 
- Hydrogen bonding energy: likelyhood of H bonds basing on the arrangememnts. More H bonds = more stable. 
- SASA: this is not strictly an energy term; to penalize proteins with overly exposed hydroiphobic residues;
- Secondary structure preservation: ensure that the secondary structures are kept. 


# Example of loss function with the proposed loss terms

In [ ]:
for batch in loader:
    # Forward pass: Get predicted positions of atoms
    predicted_positions = model(batch)  # Model output: predicted atom positions

    # Compute the energy loss for the entire batch of proteins
    total_loss = total_energy_loss(batch.edge_index, predicted_positions, 
                                   batch.ideal_bond_lengths, batch.ideal_angles, 
                                   batch.ideal_torsions, batch.secondary_structure, 
                                   batch.target_ss, batch.charges, batch.hydrophobic_residues)
    
    # Backpropagate and update the model weights
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()


In [ ]:
# Ideal Bond Lengths (example for different bond types)
ideal_bond_lengths = {
    'C-H': 1.09,  # C-H bond length in Angstroms
    'C-N': 1.32,  # C-N bond length
    'C=O': 1.22,  # C=O bond length
    'N-H': 1.01,  # N-H bond length
    # Add more bond types as needed
}

# Ideal Bond Angles (example for different angles)
ideal_angles = {
    'C-N-C': 110.0,  # C-N-C bond angle
    'N-C-O': 120.0,  # N-C-O bond angle
    # Add more angles as needed
}

# Ideal Torsions (example for different torsions)
ideal_torsions = {
    'C-N-C-N': 180.0,  # Torsion between C-N-C-N atoms
    'C-C-N-O': 60.0,   # Torsion between C-C-N-O atoms
    # Add more torsions as needed
}

# Hydrogen bond distances (example for different hydrogen bonds)
ideal_hb_distances = {
    'N-H...O': 2.8,  # Ideal N-H...O hydrogen bond distance
    'O-H...N': 2.7,  # Ideal O-H...N hydrogen bond distance
    # Add more hydrogen bonds as needed
}

# Function to get the ideal bond length for a specific bond type
def get_ideal_bond_length(bond_type):
    return ideal_bond_lengths.get(bond_type, None)

# Function to get the ideal bond angle for a specific angle type
def get_ideal_angle(angle_type):
    return ideal_angles.get(angle_type, None)

# Function to get the ideal torsion for a specific torsion type
def get_ideal_torsion(torsion_type):
    return ideal_torsions.get(torsion_type, None)

# Function to get the ideal hydrogen bond distance for a specific hydrogen bond type
def get_ideal_hb_distance(hb_type):
    return ideal_hb_distances.get(hb_type, None)


In [ ]:
def bond_loss_relative(edge_index, relative_positions, bond_types, k_b=1.0):
    bond_lengths = torch.norm(relative_positions[edge_index[0]] - relative_positions[edge_index[1]], dim=1)
    
    # Look up the ideal bond lengths based on the bond type
    ideal_lengths = torch.tensor([get_ideal_bond_length(bond_type) for bond_type in bond_types], dtype=torch.float32)

    # Compute bond loss for each bond in the graph
    bond_loss = k_b * torch.sum((bond_lengths - ideal_lengths) ** 2)
    return bond_loss


def angle_loss_relative(edge_index, relative_positions, angle_types, k_theta=1.0):
    atom_a_positions = relative_positions[edge_index[0]]
    atom_b_positions = relative_positions[edge_index[1]]
    # Compute the angle between bond vectors
    bond_vector_ab = atom_b_positions - atom_a_positions
    cos_angle = torch.cos(torch.acos(torch.sum(bond_vector_ab, dim=1)))
    
    # Look up the ideal angle for the specific bond triplet (angle type)
    ideal_angles = torch.tensor([get_ideal_angle(angle_type) for angle_type in angle_types], dtype=torch.float32)
    
    # Compute angle loss
    angle_loss = k_theta * torch.sum((cos_angle - ideal_angles) ** 2)
    return angle_loss

def torsion_loss_relative(edge_index, relative_positions, torsion_types, V_n=1.0, n=1, gamma=0.0):
    atom_a_positions = relative_positions[edge_index[0]]
    atom_b_positions = relative_positions[edge_index[1]]
    torsion_angle = torch.acos(torch.sum(atom_a_positions * atom_b_positions))
    
    # Look up the ideal torsion angle for the specific torsion type
    ideal_torsions = torch.tensor([get_ideal_torsion(torsion_type) for torsion_type in torsion_types], dtype=torch.float32)
    
    # Compute torsion loss
    torsion_loss = V_n * torch.sum((1 + torch.cos(n * torsion_angle - gamma) - ideal_torsions) ** 2)
    return torsion_loss


def hydrogen_bond_loss(atom_positions, donor_acceptor_pairs, hb_types, ideal_hb_distances, k_hb=1.0):
    loss = 0
    for i, pair in enumerate(donor_acceptor_pairs):
        donor_position = atom_positions[pair[0]]
        acceptor_position = atom_positions[pair[1]]
        distance = torch.norm(donor_position - acceptor_position)
        
        # Look up the ideal hydrogen bond distance for the current hydrogen bond type
        ideal_distance = get_ideal_hb_distance(hb_types[i])
        
        # Compute hydrogen bond loss
        loss += k_hb * (distance - ideal_distance) ** 2
    
    return loss


# SASA loss function: Penalizing high solvent exposure for hydrophobic residues
def sasa_loss(positions, hydrophobic_residues, k_sasa=1.0):
    # For simplicity, assume a pre-calculated SASA
    sasa = compute_sasa(positions)  # A function that returns precomputed SASA values
    
    loss = 0
    for idx in hydrophobic_residues:
        if sasa[idx] > 0.2:  # Threshold for exposed hydrophobic residue
            loss += k_sasa * (sasa[idx] - 0.2) ** 2
    return loss


def vdW_loss(edge_index, positions, epsilon=1.0, sigma=1.0):
    """
    Compute van der Waals loss for graph edges using Lennard-Jones potential.
    edge_index: Tensor of shape [2, num_edges] where each column represents a bond between two atoms.
    positions: Tensor of shape [num_atoms, 3] with atom coordinates in 3D space.
    epsilon: Depth of the potential well.
    sigma: Finite distance where potential is zero.
    
    Returns the van der Waals loss.
    """
    atom_a_positions = positions[edge_index[0]]
    atom_b_positions = positions[edge_index[1]]
    
    # Calculate distances between atoms
    distances = torch.norm(atom_a_positions - atom_b_positions, dim=1)
    
    # Compute van der Waals potential using Lennard-Jones potential
    r6 = distances ** 6
    r12 = distances ** 12
    vdW_interaction = epsilon * (sigma / distances) ** 12 - 2 * epsilon * (sigma / distances) ** 6
    
    # Apply van der Waals loss (penalize the interaction)
    vdW_loss = torch.sum(vdW_interaction ** 2)
    return vdW_loss

# Secondary Structure Preservation (if available as ground truth)
def secondary_structure_loss(predicted_ss, target_ss):
    """
    Compute loss based on secondary structure preservation.
    predicted_ss: Predicted secondary structure (e.g., helix, sheet).
    target_ss: Target secondary structure (e.g., helix, sheet).
    """
    # Cross-entropy loss can be used to compare predicted and target secondary structures
    return torch.nn.CrossEntropyLoss()(predicted_ss, target_ss)


In [ ]:
def total_energy_loss(batch_edge_index, batch_positions, batch_ideal_bond_lengths, batch_ideal_angles, batch_ideal_torsions, 
                      batch_secondary_structure, batch_target_ss, batch_charges, batch_hydrophobic_residues,
                      k_b=1.0, k_theta=1.0, V_n=1.0, k_e=1.0, epsilon=1.0, sigma=1.0, k_hb=1.0, k_sasa=1.0):
    
    # Compute each energy term using relative coordinates
    bond_loss_value = bond_loss_relative(batch_edge_index, batch_positions, batch_ideal_bond_lengths, k_b)
    angle_loss_value = angle_loss_relative(batch_edge_index, batch_positions, batch_ideal_angles, k_theta)
    torsion_loss_value = torsion_loss_relative(batch_edge_index, batch_positions, batch_ideal_torsions, V_n)
    secondary_structure_loss_value = secondary_structure_loss(batch_secondary_structure, batch_target_ss)
    electrostatic_loss_value = electrostatic_loss(batch_edge_index, batch_positions, batch_charges, k_e)
    vdW_loss_value = vdW_loss(batch_edge_index, batch_positions, epsilon, sigma)
    hydrogen_bond_loss_value = hydrogen_bond_loss(batch_positions, batch_hydrophobic_residues, k_hb)
    sasa_loss_value = sasa_loss(batch_positions, batch_hydrophobic_residues, k_sasa)

    # Aggregate losses
    total_loss = (bond_loss_value + angle_loss_value + torsion_loss_value +
                  secondary_structure_loss_value + electrostatic_loss_value +
                  vdW_loss_value + hydrogen_bond_loss_value + sasa_loss_value)
    
    return total_loss


### To get the "ideal values"

from Bio import PDB

# Load a PDB structure
parser = PDB.PPBuilder()
structure = parser.get_structure('Protein', 'protein.pdb')

# Extract bonds, angles, etc. (you can create custom code to extract bond lengths and angles)
for model in structure:
    for chain in model:
        for residue in chain:
            # Extracting bond lengths and angles here
            pass


In [ ]:
# Extract bond information from PDB with Biopython

from Bio import PDB

def extract_bonds_from_pdb(pdb_filename):
    parser = PDB.PPBuilder()
    structure = parser.get_structure("Protein", pdb_filename)

    bonds = []
    for model in structure:
        for chain in model:
            for residue in chain:
                # Extract the atoms of the residue
                atoms = list(residue.get_atoms())
                for i in range(len(atoms)):
                    for j in range(i + 1, len(atoms)):
                        atom1 = atoms[i]
                        atom2 = atoms[j]
                        # You can add bond type and distance information here
                        bond_type = f"{atom1.element}-{atom2.element}"  # For example, C-H, N-C, etc.
                        distance = atom1 - atom2  # Calculate the distance between the atoms
                        bonds.append((bond_type, distance))

    return bonds

# Example usage
pdb_filename = 'protein.pdb'
bonds = extract_bonds_from_pdb(pdb_filename)
for bond in bonds:
    print(bond)


In [ ]:
# 2 : look for ideal bond lenghts from force fields models AMBER or CHARMM

# Example ideal bond lengths for different atom types (in Angstroms)
ideal_bond_lengths = {
    'C-H': 1.09,  # C-H bond length in Angstroms
    'N-H': 1.01,  # N-H bond length
    'C-N': 1.32,  # C-N bond length
    'C=O': 1.22,  # C=O bond length
    # Add more bond types as needed
}

def get_ideal_bond_length(bond_type):
    return ideal_bond_lengths.get(bond_type, None)

# Example usage
bond_type = 'C-H'
ideal_length = get_ideal_bond_length(bond_type)
print(f"Ideal bond length for {bond_type}: {ideal_length} Å")


In [ ]:
# Example ideal bond angles (in degrees)
ideal_angles = {
    'C-N-C': 110.0,  # C-N-C bond angle
    'N-C-O': 120.0,  # N-C-O bond angle
    # Add more angles as needed
}

# Example ideal torsions (in degrees)
ideal_torsions = {
    'C-N-C-N': 180.0,  # Torsion between C-N-C-N atoms
    'C-C-N-O': 60.0,  # Another example torsion
    # Add more torsions as needed
}

def get_ideal_angle(angle_type):
    return ideal_angles.get(angle_type, None)

def get_ideal_torsion(torsion_type):
    return ideal_torsions.get(torsion_type, None)


### For all PDBs in one step:

from Bio import PDB
import os

def extract_bonds_from_pdb(pdb_filename):
    """
    Extract bond information from a PDB file and return bond types (atom1-element, atom2-element).
    pdb_filename: The path to the PDB file.
    
    Returns a list of bond types.
    """
    parser = PDB.PPBuilder()
    structure = parser.get_structure("Protein", pdb_filename)
    
    bonds = set()  # Use a set to store unique bonds
    
    for model in structure:
        for chain in model:
            for residue in chain:
                # Extract atoms in the residue
                atoms = list(residue.get_atoms())
                
                for i in range(len(atoms)):
                    for j in range(i + 1, len(atoms)):  # Get unique pairs of atoms
                        atom1 = atoms[i]
                        atom2 = atoms[j]
                        
                        # Define the bond type based on atom types (e.g., C-H, N-O, C-C)
                        bond_type = f"{atom1.element}-{atom2.element}"
                        
                        # Add the bond type to the set
                        bonds.add(bond_type)
    
    return bonds

def extract_bonds_from_all_pdbs(pdb_folder):
    """
    Extract unique bond types from all PDB files in a folder.
    pdb_folder: The path to the folder containing PDB files.
    
    Returns a set of unique bond types across all PDBs.
    """
    all_bonds = set()
    
    # Loop through all PDB files in the folder
    for pdb_file in os.listdir(pdb_folder):
        if pdb_file.endswith('.pdb'):
            pdb_path = os.path.join(pdb_folder, pdb_file)
            # Extract bonds from the current PDB file
            bonds = extract_bonds_from_pdb(pdb_path)
            # Add the bonds to the all_bonds set (automatically handles uniqueness)
            all_bonds.update(bonds)
    
    return all_bonds

# Example usage:
pdb_folder = 'path/to/your/pdb_folder'  # Set your PDB folder path here
unique_bonds = extract_bonds_from_all_pdbs(pdb_folder)

# Print all unique bond types
print(f"Unique bond types across all PDBs: {unique_bonds}")


### Angles and torsions

from Bio import PDB
import os

# Function to compute bond length
def compute_bond_length(atom1, atom2):
    return atom1 - atom2

# Function to compute bond angle
def compute_angle(atom1, atom2, atom3):
    # Vectors between atom1-atom2 and atom2-atom3
    vector_ab = atom2.coord - atom1.coord
    vector_bc = atom2.coord - atom3.coord
    # Calculate the cosine of the angle using dot product
    cosine_angle = torch.dot(vector_ab, vector_bc) / (torch.norm(vector_ab) * torch.norm(vector_bc))
    # Return angle in radians
    return torch.acos(cosine_angle)

# Function to compute torsion (dihedral angle)
def compute_torsion(atom1, atom2, atom3, atom4):
    # Vectors for the four atoms
    vector_a = atom2.coord - atom1.coord
    vector_b = atom3.coord - atom2.coord
    vector_c = atom4.coord - atom3.coord
    # Compute normal vectors for planes formed by (atom1, atom2, atom3) and (atom2, atom3, atom4)
    normal_1 = torch.cross(vector_a, vector_b)
    normal_2 = torch.cross(vector_b, vector_c)
    # Calculate torsion angle between normal vectors
    cosine_torsion = torch.dot(normal_1, normal_2) / (torch.norm(normal_1) * torch.norm(normal_2))
    return torch.acos(cosine_torsion)

def extract_bonds_angles_torsions_from_pdb(pdb_filename):
    """
    Extract bond information, bond angles, and torsions from a PDB file.
    pdb_filename: The path to the PDB file.

    Returns a tuple of:
    - Unique bond types (set)
    - Unique angles (set)
    - Unique torsions (set)
    """
    parser = PDB.PPBuilder()
    structure = parser.get_structure("Protein", pdb_filename)

    bonds = set()   # Set to store unique bond types
    angles = set()  # Set to store unique angles (e.g., C-N-C)
    torsions = set()  # Set to store unique torsions (e.g., C-N-C-N)

    for model in structure:
        for chain in model:
            for residue in chain:
                atoms = list(residue.get_atoms())
                
                # Extract bonds (pairwise atom interactions)
                for i in range(len(atoms)):
                    for j in range(i + 1, len(atoms)):
                        atom1 = atoms[i]
                        atom2 = atoms[j]
                        bond_type = f"{atom1.element}-{atom2.element}"
                        bonds.add(bond_type)
                
                # Extract angles (triplets of atoms)
                for i in range(len(atoms) - 2):
                    for j in range(i + 1, len(atoms) - 1):
                        for k in range(j + 1, len(atoms)):
                            atom1 = atoms[i]
                            atom2 = atoms[j]
                            atom3 = atoms[k]
                            # Compute angle between the triplet of atoms
                            angle = compute_angle(atom1, atom2, atom3)
                            angle_type = f"{atom1.element}-{atom2.element}-{atom3.element}"
                            angles.add((angle_type, angle.item()))
                
                # Extract torsions (quadruplets of atoms)
                for i in range(len(atoms) - 3):
                    for j in range(i + 1, len(atoms) - 2):
                        for k in range(j + 1, len(atoms) - 1):
                            for l in range(k + 1, len(atoms)):
                                atom1 = atoms[i]
                                atom2 = atoms[j]
                                atom3 = atoms[k]
                                atom4 = atoms[l]
                                # Compute torsion between the four atoms
                                torsion = compute_torsion(atom1, atom2, atom3, atom4)
                                torsion_type = f"{atom1.element}-{atom2.element}-{atom3.element}-{atom4.element}"
                                torsions.add((torsion_type, torsion.item()))

    return bonds, angles, torsions

def extract_all_bonds_angles_torsions(pdb_folder):
    """
    Extract bond information, angles, and torsions from all PDB files in a folder.
    pdb_folder: The path to the folder containing PDB files.
    
    Returns a tuple of:
    - Set of unique bond types
    - Set of unique angles
    - Set of unique torsions
    """
    all_bonds = set()
    all_angles = set()
    all_torsions = set()

    # Loop through all PDB files in the folder
    for pdb_file in os.listdir(pdb_folder):
        if pdb_file.endswith('.pdb'):
            pdb_path = os.path.join(pdb_folder, pdb_file)
            bonds, angles, torsions = extract_bonds_angles_torsions_from_pdb(pdb_path)
            all_bonds.update(bonds)
            all_angles.update(angles)
            all_torsions.update(torsions)

    return all_bonds, all_angles, all_torsions

# Example usage:
pdb_folder = 'path/to/your/pdb_folder'  # Set your PDB folder path here
unique_bonds, unique_angles, unique_torsions = extract_all_bonds_angles_torsions(pdb_folder)

# Print all unique bond types
print(f"Unique bond types across all PDBs: {unique_bonds}")
print(f"Unique angles across all PDBs: {unique_angles}")
print(f"Unique torsions across all PDBs: {unique_torsions}")


### Hydrogen bonds --> Graphein 

import graphein as gr
import networkx as nx
from Bio import PDB

# Function to compute hydrogen bonds
def compute_hydrogen_bonds(pdb_filename):
    # Load the PDB file using Biopython
    parser = PDB.PPBuilder()
    structure = parser.get_structure("Protein", pdb_filename)

    # Create a Graph from the structure using Graphein
    graph = gr.graphs.protein_graph(structure)

    # Compute hydrogen bonds using Graphein's built-in method
    hydrogen_bonds = gr.analysis.compute_hydrogen_bonds(graph)

    return hydrogen_bonds

# Example usage
pdb_filename = 'path_to_your_pdb_file.pdb'
hydrogen_bonds = compute_hydrogen_bonds(pdb_filename)

# Print hydrogen bonds (these will typically be pairs of atoms involved in hydrogen bonding)
print(f"Hydrogen Bonds: {hydrogen_bonds}")


### ABout the secondary structuers

Each node in protein graph (representing a residue) will have a secondary structure feature as part of its node feature vector. This feature could represent the probability distribution of the residue being part of an alpha-helix, beta-sheet, or coil, or it could directly represent the predicted secondary structure from the model during training.  
1. Self-Supervised Secondary Structure Learning
Rather than relying on target_ss from a pre-trained model, the model will implicitly learn the secondary structure by minimizing the energy function.

Here’s how we can approach this:

Implicit Secondary Structure Loss:  
  
The secondary structure can be viewed as a geometric property of the protein's backbone (e.g., torsion angles, hydrogen bonding, and bond angles that define whether the residue is part of a helix, sheet, or coil).  
  
For example, the backbone torsions (ϕ, ψ) of a residue are typically constrained in helices or sheets in specific ways, and this can be used as a form of self-supervision. So, if the model predicts a conformation that is geometrically consistent with an alpha-helix (e.g., specific values of torsion angles), it can be considered as learning the secondary structure.

Energy Minimization to Implicitly Learn Secondary Structure
Instead of using a target secondary structure (as from a database), you can define the ideal values for bond angles, torsion angles, and hydrogen bond patterns that are consistent with alpha-helix and beta-sheet geometries, and let the model learn these implicit patterns during energy minimization.

Strategy:
Bond Angle Loss: For alpha-helix prediction, use relative bond angles between consecutive residues. The ideal bond angles for helices are typically specific values based on known helical structures.

Torsion Loss: For beta-sheet and alpha-helix, the relative torsion angles (ϕ, ψ) should satisfy certain ideal values. The model can learn to optimize these torsions during training.

Hydrogen Bond Loss: Even with relative positions, hydrogen bonds are formed when certain distance and angle criteria are satisfied. You can use relative positions to define these donor-acceptor pairs and compute the hydrogen bond loss.

Self-Supervised Secondary Structure Learning: The model will learn secondary structure implicitly by optimizing the relative positions (through energy minimization), where ideal bond angles and torsions for helices and sheets serve as the target structures.



Can be seen as a combination of angles, torsion and H bond losses. 

In [ ]:
def bond_angle_loss(edge_index, relative_positions, ideal_angles, k_theta=1.0):
    loss = 0
    for i in range(len(edge_index[0]) - 2):  # Iterate through all possible triples (for angles)
        atom_a = relative_positions[edge_index[0][i]]
        atom_b = relative_positions[edge_index[1][i]]
        atom_c = relative_positions[edge_index[2][i]]
        
        # Calculate relative bond angle (this is simplified; ideally you'd calculate the angle between two vectors)
        cos_angle = torch.cos(torch.acos(torch.sum(atom_b - atom_a, dim=1) / (torch.norm(atom_b - atom_a, dim=1) * torch.norm(atom_c - atom_b, dim=1))))
        
        # Get the ideal angle for the current bond triplet (e.g., for helix or sheet)
        ideal_angle = get_ideal_angle('C-N-C')  # Replace with dynamic angle lookup
        loss += k_theta * (cos_angle - ideal_angle) ** 2

    return loss


In [ ]:
def torsion_loss_relative(edge_index, relative_positions, ideal_torsions, k_torsion=1.0):
    loss = 0
    for i in range(len(edge_index[0]) - 3):  # Iterate through all possible quadruples for torsion angles
        atom_a = relative_positions[edge_index[0][i]]
        atom_b = relative_positions[edge_index[1][i]]
        atom_c = relative_positions[edge_index[2][i]]
        atom_d = relative_positions[edge_index[3][i]]

        # Calculate relative torsion angle (simplified approach)
        torsion_angle = compute_torsion(atom_a, atom_b, atom_c, atom_d)
        
        # Get the ideal torsion angle for the current torsion type (e.g., C-N-C-N)
        ideal_torsion = get_ideal_torsion('C-N-C-N')  # Replace with dynamic torsion lookup
        loss += k_torsion * (torsion_angle - ideal_torsion) ** 2

    return loss


In [ ]:
def hydrogen_bond_loss(atom_positions, donor_acceptor_pairs, hb_types, k_hb=1.0):
    loss = 0
    for i, pair in enumerate(donor_acceptor_pairs):
        donor_position = atom_positions[pair[0]]
        acceptor_position = atom_positions[pair[1]]
        distance = torch.norm(donor_position - acceptor_position)
        
        # Look up the ideal hydrogen bond distance for the specific bond type (e.g., N-H...O)
        ideal_distance = get_ideal_hb_distance(hb_types[i])
        
        # Hydrogen bond loss (penalize distance deviations)
        loss += k_hb * (distance - ideal_distance) ** 2
    
    return loss


In [ ]:
def secondary_structure_loss(predicted_probs, target_ss):
    """
    Compute the loss comparing the predicted secondary structure probabilities with the actual secondary structure.
    predicted_probs: The predicted probabilities for each secondary structure (e.g., helix, sheet, coil).
    target_ss: The actual secondary structure labels (e.g., "H" for helix, "E" for sheet).
    """
    # Convert actual secondary structure labels to indices (0: helix, 1: sheet, 2: coil)
    label_map = {'H': 0, 'E': 1, 'C': 2}
    target_indices = torch.tensor([label_map[ss] for ss in target_ss], dtype=torch.long)
    
    # Apply cross-entropy loss: this function expects logits, not probabilities
    # You should use the logits (before softmax) in this case, but we show how to compute loss with probabilities
    loss = torch.nn.CrossEntropyLoss()(predicted_probs, target_indices)
    
    return loss


In [ ]:
def total_energy_loss(batch_edge_index, batch_positions, batch_ideal_bond_lengths, batch_ideal_angles, batch_ideal_torsions, 
                      batch_secondary_structure, batch_target_ss, batch_charges, batch_hydrophobic_residues,
                      k_b=1.0, k_theta=1.0, V_n=1.0, k_e=1.0, epsilon=1.0, sigma=1.0, k_hb=1.0, k_sasa=1.0):
    
    # Compute each energy term using relative coordinates
    bond_loss_value = bond_loss_relative(batch_edge_index, batch_positions, batch_ideal_bond_lengths, k_b)
    angle_loss_value = angle_loss_relative(batch_edge_index, batch_positions, batch_ideal_angles, k_theta)
    torsion_loss_value = torsion_loss_relative(batch_edge_index, batch_positions, batch_ideal_torsions, V_n)
    secondary_structure_loss_value = secondary_structure_loss(batch_secondary_structure, batch_target_ss)
    electrostatic_loss_value = electrostatic_loss(batch_edge_index, batch_positions, batch_charges, k_e)
    vdW_loss_value = vdW_loss(batch_edge_index, batch_positions, epsilon, sigma)
    hydrogen_bond_loss_value = hydrogen_bond_loss(batch_positions, batch_hydrophobic_residues, k_hb)
    sasa_loss_value = sasa_loss(batch_positions, batch_hydrophobic_residues, k_sasa)

    # Aggregate losses
    total_loss = (bond_loss_value + angle_loss_value + torsion_loss_value +
                  secondary_structure_loss_value + electrostatic_loss_value +
                  vdW_loss_value + hydrogen_bond_loss_value + sasa_loss_value)
    
    return total_loss


Let's break down each of these terms to clarify what they represent and how you can handle them, and address the points where you might need further guidance.

1. batch_edge_index:
Definition: This represents the indices of the residues (nodes) that are connected by a bond or interaction. Each entry in the index corresponds to a pair of residues (nodes) that have a bond or interaction.

How to obtain: You can compute this when building the graph for your protein. The graph will be defined by residues as nodes, and the bonds (covalent or non-covalent) between them will define the edges.

2. batch_positions:
Definition: These are the 3D coordinates (or relative coordinates) of each residue (or atom) in the protein. The positions could either be absolute 3D coordinates or relative distances and angles (depending on how you're working with the graph).

If you're working with relative positions, they represent the relative distance vectors between residues, meaning that each residue’s position is described relative to its neighbors or the protein’s center of mass, rather than in an absolute 3D coordinate space.

How to obtain: In a GNN, this is typically computed as part of the input graph. You will need to compute the relative positions of each node (residue) based on either structural data (like a PDB file) or through a self-supervised process.

If you're working with relative positions, you can extract them from a PDB file or calculate them by finding the pairwise distances between atoms or residues.

3. batch_ideal_bond_lengths, batch_ideal_angles, batch_ideal_torsions:
Definition: These are the ideal reference values for bond lengths, bond angles, and torsion angles. These values are used to compare the predicted values during training.

For example, the ideal bond length for a C-H bond might be 1.09 Å, the ideal bond angle for C-N-C could be 110°, and the ideal torsion angle for C-N-C-N might be 180°.

How to obtain: You would define these in dictionaries (as we discussed before) where each bond type (like C-H, C-N) maps to its ideal bond length. Similarly, you would do the same for bond angles and torsions.

4. batch_secondary_structure:
Definition: This would represent the predicted secondary structure (e.g., helix, sheet, coil) for each residue in the protein.

This is not directly provided in the PDB file but can be predicted by the model itself (during training) or inferred from relative positions, bond angles, torsions, and hydrogen bonds.

Self-supervised learning would infer the secondary structure by training the model to minimize the energy loss and maintaining geometric constraints (e.g., torsion angles).

How to obtain:

During training, batch_secondary_structure can be derived directly as part of your model's output, based on learned features (relative distances, bond angles, etc.).

If using Graphein or similar tools, the secondary structure could also be pre-computed for each residue (using Graphein's compute_secondary_structure method, for instance) and used as a target during training.

5. batch_target_ss:
Definition: These are the ground truth labels for the secondary structure of each residue. This is what you’re comparing against in your loss function.

These would typically come from experimental data (e.g., DSSP or PDB secondary structure annotations), where each residue has a label (e.g., H for helix, E for sheet, C for coil).

How to obtain:

If you don’t have secondary structure annotations for your proteins, you will need to extract this information from a PDB file (using Graphein, DSSP, or similar tools).

Alternatively, self-supervised learning (based on energy minimization) would help the model learn secondary structure as it minimizes the energy of the structure during training.

6. batch_charges:
Definition: This would be a vector of partial charges for each residue or atom in the protein. These charges are used to compute electrostatic interactions between residues (atoms) in the protein.

Electrostatic interactions depend on the relative positions of charged atoms (e.g., between charged residues like lysine and glutamate).

How to obtain: The charges can either be:

Predefined: You could use force field parameters to assign partial charges to atoms (e.g., from the AMBER, CHARMM, or OPLS force field).

Predicted: Alternatively, you could predict charges from the residue type or structural features using force field tools or machine learning models.

7. batch_hydrophobic_residues:
Definition: This would be a list of residues (or indices of residues) that are hydrophobic. The SASA loss penalizes hydrophobic residues that are exposed to the solvent.

Hydrophobic residues are typically those with nonpolar side chains (e.g., Ala, Val, Leu, Phe, Ile).

How to obtain:

You can predefine a list of hydrophobic residues based on standard classification (e.g., Hydropathy Index).

Alternatively, the hydrophobicity of a residue can be predicted using a physicochemical descriptor or a model trained on protein properties.

Summary of What You Have and Don’t Have:
What You Have:
batch_edge_index: Indices of connected residues (bonds).

batch_ideal_bond_lengths, batch_ideal_angles, batch_ideal_torsions: Predefined dictionaries with ideal values for bonds, angles, and torsions.

What You Need to Obtain:
batch_positions: These are the relative positions (not the full 3D coordinates). You need to compute them either based on PDB files or via your model’s predictions.

batch_secondary_structure: Predicted by the model based on the learned features from the training process, or precomputed using tools like Graphein.

batch_target_ss: Ground truth secondary structure labels for each residue. You can extract this from PDB files or use external tools (like DSSP, Graphein, or AlphaFold).

batch_charges: Partial charges for each residue or atom. You can extract this from force field parameters (AMBER, CHARMM) or pre-compute them.

batch_hydrophobic_residues: A list of hydrophobic residues based on predefined classification or predictions.

What Can Be Constants:
The constants k_b=1.0, k_theta=1.0, V_n=1.0, k_e=1.0, epsilon=1.0, sigma=1.0, k_hb=1.0, and k_sasa=1.0 are tunable hyperparameters that control the relative importance of different loss terms. These can remain constant or be adjusted during training to balance the contributions of each term to the total loss.

In [ ]:
# Simplified charge assignment based on amino acid type
def assign_charges(residue_type):
    charge_dict = {
        'E': -1.0,  # Glutamate (acidic, negative charge)
        'D': -1.0,  # Aspartate (acidic, negative charge)
        'K': +1.0,  # Lysine (basic, positive charge)
        'R': +1.0,  # Arginine (basic, positive charge)
        'H': +0.5,  # Histidine (slightly positive depending on pH)
        'A': 0.0,   # Alanine (neutral)
        'V': 0.0,   # Valine (neutral)
        'L': 0.0,   # Leucine (neutral)
        'I': 0.0,   # Isoleucine (neutral)
        'F': 0.0,   # Phenylalanine (neutral)
        # Add more residues as needed...
    }
    return charge_dict.get(residue_type, 0.0)  # Default to neutral if not found
